In [66]:
# =====================================================================
# 05 - TOPSIS ENTEGRASYONU İLE 2025-2030 İHTİYAÇ ÖNCELİĞİ
#
# Amaç:
#   1) Ensemble modelinden gelen 2025-2030 tahminlerini almak
#   2) 0-14 ve 65+ yaş oranlarını kullanarak demografik ihtiyaç oluşturmak
#   3) 1000 kişi başına hekim ve 100 bin kişi başına yatak göstergelerini
#      arz/kapasite göstergesi olarak kullanmak
#   4) TOPSIS ile il bazlı ihtiyaç öncelik skoru üretmek
#   5) Ensemble tahminlerini TOPSIS önceliğine göre yeniden dağıtmak
#
# Önemli mantık:
#   - Ensemble model sayı tahmini üretir.
#   - TOPSIS bu sayıları doğrudan tahmin etmez; illerin ihtiyaç önceliğini hesaplar.
#   - Son adımda ulusal toplam korunarak, artış miktarı TOPSIS skoruna göre
#     illere yeniden dağıtılır.
#
# Girdi dosyaları:
#   - model_giris.csv
#   - tahmin_2025_2030_ensemble.xlsx veya tahmin_2025_2030_ensemble.csv
#   - hamveriler\tuik26-30hesaplanmis\Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx
#   - nufus-yas.csv
#   - 1000kisibasinahekim.csv                      [opsiyonel kontrol]
#   - 100binkisibasinahastaneyatak.csv             [opsiyonel kontrol]
#
# Çıktılar:
#   - topsis_skorlari_2025_2030.csv / .xlsx
#   - topsis_duzeltilmis_ihtiyac_2025_2030.csv / .xlsx
#   - topsis_tek_dosya_2025_2030.xlsx
# =====================================================================

import os
import re
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# 0) AYARLAR
# ---------------------------------------------------------------------

MODEL_GIRIS_YOL = "model_giris.csv"

# Ensemble tahmin dosyanın adı farklıysa burayı değiştir.
ENSEMBLE_TAHMIN_ADAYLARI = [
    "tahmin_2025_2030_ensemble.xlsx",
    "tahmin_2025_2030_ensemble.csv",
    "tahmin_2025_2030_tekil.xlsx",   # gerekirse geçici alternatif
    "tahmin_2025_2030_tekil.csv",
]

TUIK_NUFUS_YOL = r"hamveriler\tuik26-30hesaplanmis\Turkiye_Nufus_Projeksiyon_Guncellenmis.xlsx"
YAS_NUFUS_YOL = "nufus-yas.csv"
HEKIM_1000_YOL = "1000kisibasinahekim.csv"
YATAK_100000_YOL = "100binkisibasinahastaneyatak.csv"

GELECEK_YILLAR = list(range(2025, 2031))
HEDEFLER = ["doktor", "ebe", "diger", "hemsire", "yatak", "hastane"]

# TOPSIS ağırlıkları
# 65+ ve 0-14: talep/fayda kriteri
# hekim ve yatak: arz/maliyet kriteri
WEIGHTS = {
    "oran_65_plus": 0.35,
    "oran_0_14": 0.10,
    "hekim_1000": 0.35,
    "yatak_100000": 0.20,
}

CRITERIA_TYPES = {
    "oran_65_plus": "benefit",
    "oran_0_14": "benefit",
    "hekim_1000": "cost",
    "yatak_100000": "cost",
}

# Çok küçük illerde aşırı sıçramayı engellemek için önerilen değerleri
# ensemble tahmininin belli oranında kırpıyoruz.
# Örn. 0.25 => TOPSIS düzeltmesi ensemble tahmininden en fazla ±%25 sapabilir.
MAX_SAPMA_ORANI = 0.25



In [68]:
# ---------------------------------------------------------------------
# 1) YARDIMCI FONKSİYONLAR
# ---------------------------------------------------------------------

def norm_text(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s


def norm_il(s):
    """İl adlarını normalize eder; Adana-1 -> Adana."""
    s = norm_text(s)
    if pd.isna(s):
        return s
    s = re.sub(r"-\d+$", "", s).strip()
    return s


def normalize_colname(c):
    """Kolon adını Türkçe karakter/boşluk farklarından arındırır."""
    c = str(c).strip().lower()
    tr_map = str.maketrans({
        "ı": "i", "İ": "i",
        "ğ": "g", "Ğ": "g",
        "ü": "u", "Ü": "u",
        "ş": "s", "Ş": "s",
        "ö": "o", "Ö": "o",
        "ç": "c", "Ç": "c"
    })
    c = c.translate(tr_map)
    c = c.replace("\u0307", "")
    c = re.sub(r"[^a-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c


def find_existing_path(paths):
    for p in paths:
        if os.path.exists(p):
            return p

    raise FileNotFoundError(
        "Tahmin dosyası bulunamadı. Şu isimlerden biri bekleniyor:\n" +
        "\n".join(paths)
    )


def standardize_health_columns(df):
    """
    Veri setindeki kolonları standart isimlere çevirir.

    Standart kolonlar:
      il_adı, yıl, nufus, doktor, ebe, diger, hemsire, yatak, hastane

    Önemli:
      Eğer doktor kolonu yoksa ama asistan + pratisyen + uzman varsa:
      doktor = asistan + pratisyen + uzman olarak oluşturulur.
    """

    df = df.copy()

    # Kolon adlarını normalize ederek eşleştirme yap
    original_cols = list(df.columns)
    norm_to_original = {normalize_colname(c): c for c in original_cols}

    rename_map = {}

    candidates = {
        "il_adı": [
            "il_adi", "il", "il_ad", "sehir", "sehir_adi", "province", "city"
        ],
        "yıl": [
            "yil", "year"
        ],
        "nufus": [
            "nufus", "toplam_nufus", "population"
        ],
        "doktor": [
            "doktor", "hekim", "toplam_hekim", "hekim_sayisi",
            "doktor_sayisi", "toplam_doktor"
        ],
        "asistan": [
            "asistan", "asistan_hekim", "asistan_sayisi", "asistan_hekim_sayisi"
        ],
        "pratisyen": [
            "pratisyen", "pratisyen_hekim", "pratisyen_sayisi",
            "pratisyen_hekim_sayisi"
        ],
        "uzman": [
            "uzman", "uzman_hekim", "uzman_sayisi", "uzman_hekim_sayisi"
        ],
        "ebe": [
            "ebe", "ebe_sayisi", "toplam_ebe"
        ],
        "diger": [
            "diger", "diger_saglik_personeli", "diger_personel",
            "diger_sayisi", "diger_saglik_personeli_sayisi",
            "diger_saglik", "diger_personel_sayisi"
        ],
        "hemsire": [
            "hemsire", "hemsire_sayisi", "toplam_hemsire"
        ],
        "yatak": [
            "yatak", "yatak_sayisi", "hastane_yatak",
            "hastane_yatak_sayisi", "toplam_yatak"
        ],
        "hastane": [
            "hastane", "hastane_sayisi", "toplam_hastane"
        ],
    }

    for std_col, possible_names in candidates.items():
        for name in possible_names:
            if name in norm_to_original:
                rename_map[norm_to_original[name]] = std_col
                break

    df = df.rename(columns=rename_map)

    if "il_adı" in df.columns:
        df["il_adı"] = df["il_adı"].apply(norm_il)

    if "yıl" in df.columns:
        df["yıl"] = pd.to_numeric(df["yıl"], errors="coerce").astype("Int64")

    numeric_cols = [
        "nufus", "doktor", "asistan", "pratisyen", "uzman",
        "ebe", "diger", "hemsire", "yatak", "hastane"
    ]

    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Doktor kolonu yoksa asistan + pratisyen + uzman toplamından oluştur
    doktor_alt_kolonlari = ["asistan", "pratisyen", "uzman"]

    if "doktor" not in df.columns and all(c in df.columns for c in doktor_alt_kolonlari):
        df["doktor"] = (
            df["asistan"].fillna(0) +
            df["pratisyen"].fillna(0) +
            df["uzman"].fillna(0)
        )

    return df


def read_model_giris(path):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    df = standardize_health_columns(df)

    required = ["il_adı", "yıl"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"model_giris.csv içinde zorunlu kolonlar eksik: {missing}\n"
            f"Mevcut kolonlar: {df.columns.tolist()}"
        )

    df = df.dropna(subset=["il_adı", "yıl"]).copy()
    df["yıl"] = df["yıl"].astype(int)

    return df


def read_population_projection(path):
    """
    TÜİK projeksiyon Excel dosyasından 2025-2030 toplam nüfusunu okur.

    Beklenen yapı:
      sheet_name = 'ML Formatı'
      header = 1
      kolonlar = Yıl, İl, Nüfus
    """

    try:
        df = pd.read_excel(path, sheet_name="ML Formatı", header=1)
    except Exception:
        df = pd.read_excel(path, header=1)

    df.columns = [str(c).strip() for c in df.columns]

    norm_map = {c: normalize_colname(c) for c in df.columns}

    yil_col = None
    il_col = None
    nufus_col = None

    for original, normed in norm_map.items():
        if normed in ["yil", "year"]:
            yil_col = original
        elif normed in ["il", "il_adi", "province", "city"]:
            il_col = original
        elif normed in ["nufus", "population", "toplam_nufus"]:
            nufus_col = original

    if yil_col is None or il_col is None or nufus_col is None:
        raise ValueError(
            "Nüfus projeksiyon dosyasında kolonlar otomatik bulunamadı.\n"
            f"Bulunan kolonlar: {df.columns.tolist()}\n"
            f"Eşleşmeler: yil_col={yil_col}, il_col={il_col}, nufus_col={nufus_col}"
        )

    out = df[[yil_col, il_col, nufus_col]].copy()
    out.columns = ["yıl", "il_adı", "nufus"]

    out["il_adı"] = out["il_adı"].apply(norm_il)
    out["yıl"] = pd.to_numeric(out["yıl"], errors="coerce")
    out["nufus"] = pd.to_numeric(out["nufus"], errors="coerce")

    out = out[out["yıl"].between(2025, 2030)].dropna(
        subset=["yıl", "il_adı", "nufus"]
    )

    out["yıl"] = out["yıl"].astype(int)
    out["nufus"] = out["nufus"].astype(float)

    out = (
        out.groupby(["il_adı", "yıl"], as_index=False)["nufus"]
        .mean()
        .sort_values(["il_adı", "yıl"])
        .reset_index(drop=True)
    )

    beklenen = 81 * 6
    if len(out) != beklenen:
        print(f"UYARI: Nüfus projeksiyonu {beklenen} satır beklenirken {len(out)} satır geldi.")
        print(out.head(10).to_string(index=False))

    return out


def read_future_ensemble(path):
    """
    2025-2030 ensemble tahmin dosyasını okur.

    Desteklenen formatlar:
      1) Wide:
         il_adı, yıl, nufus, doktor, ebe, diger, hemsire, yatak, hastane

      2) Long:
         il_adı, yıl, hedef, tahmin
    """

    if path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    df.columns = [str(c).strip() for c in df.columns]

    # Long format ise önce pivotla
    norm_cols = [normalize_colname(c) for c in df.columns]

    if "hedef" in norm_cols and "tahmin" in norm_cols:
        col_map = {normalize_colname(c): c for c in df.columns}

        il_col = col_map.get("il_adi", col_map.get("il"))
        yil_col = col_map.get("yil")
        hedef_col = col_map.get("hedef")
        tahmin_col = col_map.get("tahmin")

        if il_col is None or yil_col is None:
            raise ValueError(
                f"Long format tahmin dosyasında il/yıl kolonu bulunamadı. Kolonlar: {df.columns.tolist()}"
            )

        df = df[[il_col, yil_col, hedef_col, tahmin_col]].copy()
        df.columns = ["il_adı", "yıl", "hedef", "tahmin"]

        df["il_adı"] = df["il_adı"].apply(norm_il)
        df["yıl"] = pd.to_numeric(df["yıl"], errors="coerce").astype(int)
        df["hedef"] = df["hedef"].apply(lambda x: normalize_colname(x))

        hedef_rename = {
            "hekim": "doktor",
            "doktor": "doktor",
            "ebe": "ebe",
            "diger": "diger",
            "diger_saglik_personeli": "diger",
            "hemsire": "hemsire",
            "yatak": "yatak",
            "hastane": "hastane"
        }

        df["hedef"] = df["hedef"].map(lambda x: hedef_rename.get(x, x))

        df = (
            df.pivot_table(
                index=["il_adı", "yıl"],
                columns="hedef",
                values="tahmin",
                aggfunc="mean"
            )
            .reset_index()
        )

        df.columns.name = None

    # Wide format standardizasyonu
    df = standardize_health_columns(df)

    required = ["il_adı", "yıl"]
    missing_required = [c for c in required if c not in df.columns]

    if missing_required:
        raise ValueError(
            f"Gelecek tahmin dosyasında zorunlu kolonlar eksik: {missing_required}\n"
            f"Mevcut kolonlar: {df.columns.tolist()}"
        )

    df["il_adı"] = df["il_adı"].apply(norm_il)
    df["yıl"] = pd.to_numeric(df["yıl"], errors="coerce").astype(int)

    df = df[df["yıl"].between(2025, 2030)].copy()

    missing_targets = [h for h in HEDEFLER if h not in df.columns]

    if missing_targets:
        raise ValueError(
            "Gelecek tahmin dosyasında şu hedef kolonları yok: " +
            ", ".join(missing_targets) +
            f"\nMevcut kolonlar: {df.columns.tolist()}\n"
            "Beklenen kolonlar: il_adı, yıl, doktor, ebe, diger, hemsire, yatak, hastane"
        )

    for h in HEDEFLER:
        df[h] = pd.to_numeric(df[h], errors="coerce")

    return df.reset_index(drop=True)


def read_age_population(path):
    """
    nufus-yas.csv dosyasını okur.

    Dosya yapısı TÜİK pivot çıktısı gibi pipe ayrımlı olduğu için manuel parse edilir.

    Çıktı:
      yıl, il_adı, nufus_0_14, nufus_65_plus
    """

    rows = []
    current_year = None

    with open(path, encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")

            if len(parts) < 11:
                continue

            if re.fullmatch(r"20\d{2}", parts[0].strip()):
                current_year = int(parts[0].strip())
                city = parts[1]

            elif parts[0].strip() == "" and current_year is not None and parts[1].strip():
                city = parts[1]

            else:
                continue

            vals = []

            for p in parts[2:11]:
                try:
                    vals.append(float(str(p).replace(",", ".")))
                except Exception:
                    vals.append(np.nan)

            if len(vals) == 9 and not all(pd.isna(v) for v in vals):
                # Sıra:
                # 0-4, 10-14, 5-9, 65-69, 70-74, 75-79, 80-84, 85-89, 90+
                nufus_0_14 = np.nansum(vals[0:3])
                nufus_65_plus = np.nansum(vals[3:9])

                rows.append({
                    "yıl": current_year,
                    "il_adı": norm_il(city),
                    "nufus_0_14": nufus_0_14,
                    "nufus_65_plus": nufus_65_plus,
                })

    out = pd.DataFrame(rows)

    if out.empty:
        raise ValueError(
            "nufus-yas.csv okunamadı veya beklenen pipe ayrımlı yapı bulunamadı."
        )

    out["yıl"] = out["yıl"].astype(int)

    return out


def forecast_age_ratios(age_df, hist_total_pop, future_pop):
    """
    2007-2024 yaş grubu verilerinden il bazlı 0-14 ve 65+ oranlarını hesaplar.
    Sonra bu oranları lineer trend ile 2025-2030'a taşır.

    2002-2006 yaş verisine gerek yoktur.
    """

    hist_total_pop = standardize_health_columns(hist_total_pop)

    if "nufus" not in hist_total_pop.columns:
        raise ValueError(
            "Geçmiş model verisinde toplam nüfus kolonu bulunamadı.\n"
            f"Mevcut kolonlar: {hist_total_pop.columns.tolist()}"
        )

    hist = age_df.merge(
        hist_total_pop[["il_adı", "yıl", "nufus"]],
        on=["il_adı", "yıl"],
        how="left"
    )

    hist = hist.dropna(subset=["nufus"]).copy()

    hist["oran_0_14"] = hist["nufus_0_14"] / hist["nufus"]
    hist["oran_65_plus"] = hist["nufus_65_plus"] / hist["nufus"]

    preds = []

    for il, g in hist.groupby("il_adı"):
        g = g.sort_values("yıl")

        for oran_col in ["oran_0_14", "oran_65_plus"]:
            sub = g[["yıl", oran_col]].dropna()

            if len(sub) >= 3:
                x = sub["yıl"].values.astype(float)
                y = sub[oran_col].values.astype(float)

                coef = np.polyfit(x, y, deg=1)
                pred_vals = np.polyval(coef, GELECEK_YILLAR)

            else:
                last = sub[oran_col].iloc[-1] if len(sub) else np.nan
                pred_vals = np.repeat(last, len(GELECEK_YILLAR))

            for yil, val in zip(GELECEK_YILLAR, pred_vals):
                preds.append({
                    "il_adı": il,
                    "yıl": yil,
                    oran_col: val
                })

    p = pd.DataFrame(preds)

    p = (
        p.pivot_table(
            index=["il_adı", "yıl"],
            values=["oran_0_14", "oran_65_plus"],
            aggfunc="first"
        )
        .reset_index()
    )

    p["oran_0_14"] = p["oran_0_14"].clip(lower=0.05, upper=0.45)
    p["oran_65_plus"] = p["oran_65_plus"].clip(lower=0.02, upper=0.35)

    toplam = p["oran_0_14"] + p["oran_65_plus"]
    mask = toplam > 0.80

    p.loc[mask, ["oran_0_14", "oran_65_plus"]] = (
        p.loc[mask, ["oran_0_14", "oran_65_plus"]]
        .div(toplam[mask], axis=0) * 0.80
    )

    out = future_pop.merge(p, on=["il_adı", "yıl"], how="left")

    out["nufus_0_14_tahmin"] = out["nufus"] * out["oran_0_14"]
    out["nufus_65_plus_tahmin"] = out["nufus"] * out["oran_65_plus"]

    return out


def topsis_score(df, criteria, weights, types):
    """
    TOPSIS skoru hesaplar.

    Yüksek skor = daha yüksek ihtiyaç önceliği.
    """

    X = df[criteria].astype(float).copy()

    for c in criteria:
        X[c] = X[c].fillna(X[c].median())

    denom = np.sqrt((X ** 2).sum(axis=0))
    denom = denom.replace(0, np.nan)

    R = X / denom
    R = R.fillna(0)

    W = np.array([weights[c] for c in criteria])

    V = R.values * W

    ideal_pos = []
    ideal_neg = []

    for j, c in enumerate(criteria):
        if types[c] == "benefit":
            ideal_pos.append(np.max(V[:, j]))
            ideal_neg.append(np.min(V[:, j]))

        elif types[c] == "cost":
            ideal_pos.append(np.min(V[:, j]))
            ideal_neg.append(np.max(V[:, j]))

        else:
            raise ValueError(f"Bilinmeyen kriter tipi: {types[c]}")

    ideal_pos = np.array(ideal_pos)
    ideal_neg = np.array(ideal_neg)

    d_pos = np.sqrt(((V - ideal_pos) ** 2).sum(axis=1))
    d_neg = np.sqrt(((V - ideal_neg) ** 2).sum(axis=1))

    score = d_neg / (d_pos + d_neg + 1e-12)

    return score


def add_topsis_scores(future_df):
    criteria = list(WEIGHTS.keys())
    rows = []

    for yil, g in future_df.groupby("yıl"):
        gg = g.copy()

        gg["topsis_skor"] = topsis_score(
            gg,
            criteria,
            WEIGHTS,
            CRITERIA_TYPES
        )

        gg["topsis_sira"] = (
            gg["topsis_skor"]
            .rank(ascending=False, method="dense")
            .astype(int)
        )

        gg["ihtiyac_payi"] = gg["topsis_skor"] * gg["nufus"]

        if gg["ihtiyac_payi"].sum() > 0:
            gg["ihtiyac_payi_norm"] = gg["ihtiyac_payi"] / gg["ihtiyac_payi"].sum()
        else:
            gg["ihtiyac_payi_norm"] = 1 / len(gg)

        gg["oncelik_grubu"] = pd.qcut(
            gg["topsis_skor"].rank(method="first"),
            q=4,
            labels=["Düşük", "Orta", "Yüksek", "Çok yüksek"]
        )

        rows.append(gg)

    return pd.concat(rows, ignore_index=True)

def redistribute_by_topsis(future_with_topsis, hist_df):
    """
    Ensemble tahminlerini TOPSIS'e göre düzeltir.

    DÜZELTİLMİŞ MANTIK:
      - 2024 gerçek değeri başlangıç kabul edilir.
      - Eğer future tahmininde bir hedef değişken NaN geldiyse,
        o ilin 2024 gerçek değeriyle doldurulur.
      - Böylece hemsire gibi kolonlar NaN kalmaz.
      - Her gelecek yıl için ulusal toplam korunur.
      - Ulusal artış TOPSIS ihtiyaç payına göre illere dağıtılır.
    """

    hist_df = standardize_health_columns(hist_df)
    future_with_topsis = standardize_health_columns(future_with_topsis)

    missing_hist = [
        c for c in ["il_adı", "yıl"] + HEDEFLER
        if c not in hist_df.columns
    ]

    if missing_hist:
        raise ValueError(
            "Geçmiş veri içinde gerekli kolonlar eksik:\n"
            f"Eksik kolonlar: {missing_hist}\n"
            f"Mevcut kolonlar: {hist_df.columns.tolist()}"
        )

    missing_future = [
        c for c in ["il_adı", "yıl", "nufus", "topsis_skor", "ihtiyac_payi_norm"] + HEDEFLER
        if c not in future_with_topsis.columns
    ]

    if missing_future:
        raise ValueError(
            "Gelecek TOPSIS verisi içinde gerekli kolonlar eksik:\n"
            f"Eksik kolonlar: {missing_future}\n"
            f"Mevcut kolonlar: {future_with_topsis.columns.tolist()}"
        )

    base_2024 = hist_df.loc[
        hist_df["yıl"] == 2024,
        ["il_adı"] + HEDEFLER
    ].copy()

    if base_2024.empty:
        raise ValueError("Geçmiş veri içinde 2024 yılı bulunamadı.")

    out = future_with_topsis.merge(
        base_2024,
        on="il_adı",
        how="left",
        suffixes=("", "_2024")
    )

    # 2024 baz değerleri eksik mi kontrol et
    for hedef in HEDEFLER:
        base_col = f"{hedef}_2024"

        if base_col not in out.columns:
            raise ValueError(f"{base_col} kolonu oluşmadı. Merge işleminde sorun var.")

        eksik_base = out[base_col].isna().sum()
        if eksik_base > 0:
            print(f"UYARI: {hedef} için {eksik_base} satırda 2024 baz değeri eksik.")

    # Gelecek tahminlerinde NaN varsa 2024 değeri ile doldur
    for hedef in HEDEFLER:
        base_col = f"{hedef}_2024"

        nan_count = out[hedef].isna().sum()

        if nan_count > 0:
            print(
                f"UYARI: {hedef} gelecek tahmininde {nan_count} NaN vardı. "
                f"Bu değerler 2024 gerçek değeriyle dolduruldu."
            )

            out[hedef] = out[hedef].fillna(out[base_col])

    # Hâlâ NaN varsa 0'a değil, yıl/kolon medyanına doldur
    for hedef in HEDEFLER:
        if out[hedef].isna().sum() > 0:
            out[hedef] = out.groupby("yıl")[hedef].transform(
                lambda s: s.fillna(s.median())
            )

    for hedef in HEDEFLER:
        raw_col = hedef
        base_col = f"{hedef}_2024"
        adj_col = f"{hedef}_topsis_ihtiyac"
        fark_col = f"{hedef}_topsis_fark"

        out[adj_col] = np.nan

        for yil, idx in out.groupby("yıl").groups.items():
            sub = out.loc[idx].copy()

            # Bu aşamada raw ve base kesin sayısal olsun
            sub[raw_col] = pd.to_numeric(sub[raw_col], errors="coerce")
            sub[base_col] = pd.to_numeric(sub[base_col], errors="coerce")

            # Yıl içindeki eksikleri tekrar güvenli doldur
            sub[raw_col] = sub[raw_col].fillna(sub[base_col])
            sub[raw_col] = sub[raw_col].fillna(sub[raw_col].median())
            sub[base_col] = sub[base_col].fillna(sub[base_col].median())

            national_ensemble_total = sub[raw_col].sum(skipna=True)
            national_2024_total = sub[base_col].sum(skipna=True)
            national_delta = national_ensemble_total - national_2024_total

            share = sub["ihtiyac_payi_norm"].values.astype(float)
            base_vals = sub[base_col].values.astype(float)
            ensemble_vals = sub[raw_col].values.astype(float)

            # share bozulursa eşit dağıt
            if np.isnan(share).any() or share.sum() == 0:
                share = np.repeat(1 / len(sub), len(sub))
            else:
                share = share / share.sum()

            if national_delta >= 0:
                proposed = base_vals + national_delta * share
            else:
                inv_share = 1 - share

                if inv_share.sum() == 0:
                    inv_share = np.repeat(1 / len(sub), len(sub))
                else:
                    inv_share = inv_share / inv_share.sum()

                proposed = base_vals + national_delta * inv_share

            # Ensemble değeri NaN olmasın diye yukarıda dolduruldu
            lower = ensemble_vals * (1 - MAX_SAPMA_ORANI)
            upper = ensemble_vals * (1 + MAX_SAPMA_ORANI)

            proposed = np.clip(proposed, lower, upper)

            out.loc[idx, adj_col] = proposed

        out[adj_col] = pd.to_numeric(out[adj_col], errors="coerce")

        # Hâlâ eksik kalırsa direkt ensemble tahminiyle doldur
        out[adj_col] = out[adj_col].fillna(out[raw_col])

        out[adj_col] = out[adj_col].round().astype("Int64")
        out[fark_col] = out[adj_col].astype(float) - out[raw_col].astype(float)

    return out

In [70]:
# ---------------------------------------------------------------------
# 2) VERİ OKUMA
# ---------------------------------------------------------------------

print("========== 1) VERİ OKUMA ==========")

hist = read_model_giris(MODEL_GIRIS_YOL)

print("Geçmiş model verisi:", hist.shape, f"{hist['yıl'].min()}-{hist['yıl'].max()}")
print("Hist kolonları:", hist.columns.tolist())

eksik_hist = [c for c in ["il_adı", "yıl"] + HEDEFLER if c not in hist.columns]
print("Hist eksik kolonlar:", eksik_hist)

future_pred_path = find_existing_path([
    "tahmin_2025_2030_ensemble.xlsx",
    "tahmin_2025_2030_ensemble.csv",
    "tahmin_2025_2030_tekil.xlsx",
    "tahmin_2025_2030_tekil.csv",
])

future_pred = read_future_ensemble(future_pred_path)

print("Gelecek tahmini:", future_pred.shape, "| Dosya:", future_pred_path)
print("Gelecek tahmin kolonları:", future_pred.columns.tolist())

future_pop = read_population_projection(TUIK_NUFUS_YOL)

print("2025-2030 nüfus projeksiyonu:", future_pop.shape)

age_hist = read_age_population(YAS_NUFUS_YOL)

print("Yaş nüfus geçmiş verisi:", age_hist.shape)

========== 1) VERİ OKUMA ==========
Geçmiş model verisi: (1863, 12) 2002-2024
Hist kolonları: ['il_adı', 'yıl', 'hastane', 'yatak', 'nufus', 'asistan', 'diger', 'ebe', 'hemsire', 'pratisyen', 'uzman', 'doktor']
Hist eksik kolonlar: []
Gelecek tahmini: (486, 9) | Dosya: tahmin_2025_2030_tekil.xlsx
Gelecek tahmin kolonları: ['il_adı', 'yıl', 'nufus', 'doktor', 'ebe', 'diger', 'hemsire', 'yatak', 'hastane']
2025-2030 nüfus projeksiyonu: (486, 3)
Yaş nüfus geçmiş verisi: (1458, 4)


In [71]:
# ---------------------------------------------------------------------
# 3) 2025-2030 YAŞ ORANLARINI OLUŞTUR
# ---------------------------------------------------------------------

print("\n========== 2) YAŞ ORANLARI 2025-2030 ==========")

age_future = forecast_age_ratios(
    age_df=age_hist,
    hist_total_pop=hist[["il_adı", "yıl", "nufus"]],
    future_pop=future_pop
)

print(age_future.head(10).to_string(index=False))




========== 2) YAŞ ORANLARI 2025-2030 ==========
  il_adı  yıl     nufus  oran_0_14  oran_65_plus  nufus_0_14_tahmin  nufus_65_plus_tahmin
   Adana 2025 2283609.0   0.228675      0.100415      522204.894449         229309.046286
   Adana 2026 2293499.0   0.225789      0.103256      517846.442580         236816.928762
   Adana 2027 2299727.0   0.222902      0.106096      512614.628377         243992.472558
   Adana 2028 2305551.0   0.220016      0.108937      507257.974220         251159.387649
   Adana 2029 2310973.0   0.217130      0.111777      501780.412735         258314.454855
   Adana 2030 2315809.0   0.214243      0.114618      496146.004249         265433.158791
Adıyaman 2025  617821.0   0.265830      0.090233      164235.338593          55747.876115
Adıyaman 2026  623112.0   0.262283      0.092231      163431.766546          57470.430154
Adıyaman 2027  629218.0   0.258736      0.094230      162801.531917          59290.926710
Adıyaman 2028  634858.0   0.255189      0.096228   

In [73]:
# ---------------------------------------------------------------------
# 4) ENSEMBLE TAHMİNLERİ + YAŞ ORANLARI + KAPASİTE KRİTERLERİ
# ---------------------------------------------------------------------

print("\n========== 3) TOPSIS KRİTER MATRİSİ ==========")

future = future_pred.merge(
    age_future[["il_adı", "yıl", "nufus", "oran_0_14", "oran_65_plus",
                "nufus_0_14_tahmin", "nufus_65_plus_tahmin"]],
    on=["il_adı", "yıl"],
    how="left",
    suffixes=("", "_proj")
)

# Eğer ensemble dosyasında nufus varsa projeksiyon nüfusunu esas al
if "nufus_proj" in future.columns:
    future["nufus"] = future["nufus_proj"].combine_first(future.get("nufus"))

# Arz/kapsam göstergeleri: gelecek tahmininden kişi başı değer üret
future["hekim_1000"] = (future["doktor"] / future["nufus"]) * 1000
future["yatak_100000"] = (future["yatak"] / future["nufus"]) * 100000

criteria_cols = ["oran_65_plus", "oran_0_14", "hekim_1000", "yatak_100000"]
print(future[["il_adı", "yıl", "nufus"] + criteria_cols].head(10).to_string(index=False))




========== 3) TOPSIS KRİTER MATRİSİ ==========
  il_adı  yıl     nufus  oran_65_plus  oran_0_14  hekim_1000  yatak_100000
   Adana 2025 2283609.0      0.100415   0.228675    2.586681    319.459925
   Adana 2026 2293499.0      0.103256   0.225789    2.635606    322.061262
   Adana 2027 2299727.0      0.106096   0.222902    2.688385    325.115923
   Adana 2028 2305551.0      0.108937   0.220016    2.741359    328.170836
   Adana 2029 2310973.0      0.111777   0.217130    2.794552    331.227738
   Adana 2030 2315809.0      0.114618   0.214243    2.848216    334.315184
Adıyaman 2025  617821.0      0.090233   0.265830    1.610347    175.940534
Adıyaman 2026  623112.0      0.092231   0.262283    1.641458    174.446579
Adıyaman 2027  629218.0      0.094230   0.258736    1.669880    172.753730
Adıyaman 2028  634858.0      0.096228   0.255189    1.699001    171.219007


In [76]:
# ---------------------------------------------------------------------
# 5) TOPSIS SKORU
# ---------------------------------------------------------------------

print("\n========== 4) TOPSIS SKORLARI ==========")

future_topsis = add_topsis_scores(future)

print(
    future_topsis.sort_values(["yıl", "topsis_sira"])
    [["yıl", "topsis_sira", "il_adı", "topsis_skor", "oncelik_grubu"] + criteria_cols]
    .head(20)
    .to_string(index=False)
)




========== 4) TOPSIS SKORLARI ==========
 yıl  topsis_sira     il_adı  topsis_skor oncelik_grubu  oran_65_plus  oran_0_14  hekim_1000  yatak_100000
2025            1      Sinop     0.816062    Çok yüksek      0.211717   0.144825    1.539555    297.493079
2025            2     Artvin     0.805981    Çok yüksek      0.187289   0.143470    1.714907    220.958460
2025            3    Çankırı     0.797180    Çok yüksek      0.177340   0.158733    1.625282    236.446436
2025            4  Kastamonu     0.783555    Çok yüksek      0.200871   0.142426    1.806087    305.184973
2025            5     Bartın     0.767338    Çok yüksek      0.166232   0.142996    1.636443    245.353172
2025            6     Amasya     0.753433    Çok yüksek      0.168272   0.164245    1.795180    275.073967
2025            7     Burdur     0.745231    Çok yüksek      0.168000   0.152384    1.837195    277.751726
2025            8    Ardahan     0.743358    Çok yüksek      0.153428   0.178344    1.721591    232.48

In [78]:
print("\n========== 5) TOPSIS DÜZELTİLMİŞ İHTİYAÇ ==========")

result = redistribute_by_topsis(future_topsis, hist)

show_cols = [
    "il_adı", "yıl", "nufus", "topsis_sira", "topsis_skor", "oncelik_grubu",
    "doktor", "doktor_topsis_ihtiyac", "doktor_topsis_fark",
    "hemsire", "hemsire_topsis_ihtiyac", "hemsire_topsis_fark",
    "ebe", "ebe_topsis_ihtiyac", "ebe_topsis_fark",
    "yatak", "yatak_topsis_ihtiyac", "yatak_topsis_fark",
]

print(result.sort_values(["yıl", "topsis_sira"])[show_cols].head(20).to_string(index=False))


========== 5) TOPSIS DÜZELTİLMİŞ İHTİYAÇ ==========
UYARI: hemsire gelecek tahmininde 486 NaN vardı. Bu değerler 2024 gerçek değeriyle dolduruldu.
    il_adı  yıl     nufus  topsis_sira  topsis_skor oncelik_grubu      doktor  doktor_topsis_ihtiyac  doktor_topsis_fark  hemsire  hemsire_topsis_ihtiyac  hemsire_topsis_fark         ebe  ebe_topsis_ihtiyac  ebe_topsis_fark       yatak  yatak_topsis_ihtiyac  yatak_topsis_fark
     Sinop 2025  225848.0            1     0.816062    Çok yüksek  347.705402                    391           43.294598    831.0                     831                  0.0  224.981531                 235        10.018469  671.882170                   673           1.117830
    Artvin 2025  167531.0            2     0.805981    Çok yüksek  287.300114                    312           24.699886    473.0                     473                  0.0  196.028233                 207        10.971767  370.173918                   381          10.826082
   Çankırı 2025  2005

In [80]:
# ---------------------------------------------------------------------
# 7) ÇIKTI
# ---------------------------------------------------------------------

print("\n========== 6) ÇIKTI ==========")

score_cols = [
    "il_adı", "yıl", "nufus",
    "nufus_0_14_tahmin", "nufus_65_plus_tahmin",
    "oran_0_14", "oran_65_plus", "hekim_1000", "yatak_100000",
    "topsis_skor", "topsis_sira", "oncelik_grubu", "ihtiyac_payi_norm"
]

need_cols = ["il_adı", "yıl", "nufus", "topsis_skor", "topsis_sira", "oncelik_grubu"]
for h in HEDEFLER:
    need_cols += [h, f"{h}_topsis_ihtiyac", f"{h}_topsis_fark"]

scores_out = result[score_cols].sort_values(["yıl", "topsis_sira"]).reset_index(drop=True)
needs_out = result[need_cols].sort_values(["il_adı", "yıl"]).reset_index(drop=True)

scores_out.to_csv("topsis_skorlari_2025_2030.csv", index=False, encoding="utf-8-sig")
needs_out.to_csv("topsis_duzeltilmis_ihtiyac_2025_2030.csv", index=False, encoding="utf-8-sig")

scores_out.to_excel("topsis_skorlari_2025_2030.xlsx", index=False)
needs_out.to_excel("topsis_duzeltilmis_ihtiyac_2025_2030.xlsx", index=False)

with pd.ExcelWriter("topsis_tek_dosya_2025_2030.xlsx") as writer:
    scores_out.to_excel(writer, sheet_name="TOPSIS_Skorlari", index=False)
    needs_out.to_excel(writer, sheet_name="Duzeltilmis_Ihtiyac", index=False)
    pd.DataFrame({
        "Kriter": list(WEIGHTS.keys()),
        "Ağırlık": list(WEIGHTS.values()),
        "Tip": [CRITERIA_TYPES[k] for k in WEIGHTS.keys()]
    }).to_excel(writer, sheet_name="Kriter_Agirliklari", index=False)

print("[OK] topsis_skorlari_2025_2030.csv / .xlsx")
print("[OK] topsis_duzeltilmis_ihtiyac_2025_2030.csv / .xlsx")
print("[OK] topsis_tek_dosya_2025_2030.xlsx")
print("\nNot: Yüksek TOPSIS skoru = daha yüksek ihtiyaç önceliği.")


========== 6) ÇIKTI ==========
[OK] topsis_skorlari_2025_2030.csv / .xlsx
[OK] topsis_duzeltilmis_ihtiyac_2025_2030.csv / .xlsx
[OK] topsis_tek_dosya_2025_2030.xlsx

Not: Yüksek TOPSIS skoru = daha yüksek ihtiyaç önceliği.
